In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l3.1 Queries, keys and values

> 🎯 **Goal:** Understand attention as a soft dictionary lookup where each token produces a query, a key, and a value.

**Why this matters.** Attention is the heart of the transformer. Every later piece (scaling, masking, multi-head, the full block) is a variation on the query/key/value mechanism you meet here. Get this right and the rest of the unit falls into place.

**The intuition.** Think of a regular Python dictionary lookup: `prices["apple"]`. You hand over a key ("apple"), the dictionary finds the exact matching entry, and hands back the value. Attention is the *soft* version of this. Instead of one exact match, every token compares its **query** (what am I looking for?) against every token's **key** (what do I offer?), and gets back a blended mix of all the **values** (what each token hands over), weighted by how well query and key agree.

**The mechanics.** We start from token vectors `x` and project each token three times with three separate weight matrices:
- **Query** (`q = w_q(x)`): what this token wants to gather from the sequence.
- **Key** (`k = w_k(x)`): how this token advertises itself to others.
- **Value** (`v = w_v(x)`): the actual content this token will contribute if attended to.

The match score for token i looking at token j is the dot product `query_i . key_j`: a large number means "these line up, pay attention." Those scores are turned into a probability distribution (one row per token, summing to 1) and used to take a weighted average of the values. Note the projections are separate matrices, so a token can ask for one thing (query) while advertising another (key), which is exactly why attention is expressive.

In [ ]:
from torch import nn
from pocketlm import scaled_dot_product_attention

torch.manual_seed(0)
tokens, dim, hs = 3, 8, 8
x = torch.randn(tokens, dim)
w_q = nn.Linear(dim, hs, bias=False)
w_k = nn.Linear(dim, hs, bias=False)
w_v = nn.Linear(dim, hs, bias=False)
q, k, v = w_q(x), w_k(x), w_v(x)
out, weights = scaled_dot_product_attention(
    q.unsqueeze(0), k.unsqueeze(0), v.unsqueeze(0))
print("attention weights (each row sums to 1):")
print(weights[0].detach().round(decimals=2))

**What just happened.** The printed matrix has one row per token, and each row is that token's distribution of attention over all tokens. Every row sums to 1 because it is a probability distribution (that is what `softmax` guarantees). The output, not printed here, is the weighted sum of the value vectors: a custom blend of information that each token assembled for itself. With a random seed and untrained weights the distributions look fairly flat; after training they sharpen into meaningful patterns.

⚠️ **Common pitfalls**
- Confusing the three roles. Query, key, and value come from three *different* matrices. They are not the same vector wearing different hats.
- Expecting attention weights to be symmetric. Token i attending to token j tells you nothing about how much j attends to i; the score matrix is generally not symmetric.
- Forgetting the batch dimension. We `unsqueeze(0)` to add a leading batch axis because `scaled_dot_product_attention` expects batched input.

🏋️ **Try it yourself.** Change `tokens` from 3 to 6 and rerun. The weight matrix grows to 6x6 and every row still sums to 1. Then try setting all three weight matrices to the identity-like behavior by seeding differently, and watch how the attention pattern shifts.

In [ ]:
assert torch.allclose(weights.sum(-1), torch.ones(1, tokens), atol=1e-6)